# Data lifecycle (clean old assets)
https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-156

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
init_demo()
# Reload the global vars again
from resources.utils import *  

from resources.dask_clusters.dask_main_env import *
await init_dask_cluster_staging()

In [ ]:
# Other imports
import copy
import json
import pystac

from rs_common import prefect_utils
from rs_common.config import DATETIME_FORMAT_MS

In [ ]:
pystac_items: list[pystac.Item] = []

# Create a test collection
col_name = "collection_lifecycle"
print(f"Stage items in {col_name!r}")
create_test_collection(col_name)

# Stage auxip and cadip products
pystac_items.extend(stage_test_objects(auxip_client, 10, col_name).items)
pystac_items.extend(stage_test_objects(cadip_client, 200, col_name).items)
    
# Convert the items to dicts and sort them
def items_to_dicts(pystac_items: list[pystac.Item]) -> list[dict]:
    l = [item.to_dict() for item in pystac_items]
    l.sort(key=lambda item: item.get("collection", "") + item.get("id", ""))
    return l
original_items: list[dict] = items_to_dicts(pystac_items)

# timestamps_for_items.ipynb
for item in original_items:
    properties = item["properties"]
    assert "expires" in properties
    assert "updated" in properties
    assert "published" in properties

original_updated = {
    item["id"]: item["properties"]["updated"]
    for item in original_items
}

In [ ]:
def get_files(item: dict) -> list[str]:
    return [asset["href"] for asset in item.get('assets', {}).values()]

def print_items(items: list[Item]):
    print("Staged items:")
    for item in items:
        assets = get_files(item)
        info = {
            "id": item.get("id"),
            "properties": {
                "expires": item["properties"]["expires"],
                "updated": item["properties"].get("updated", None),
                "unpublished": item["properties"].get("unpublished", None),
            },
            "assets files": assets,
        }
        print(json.dumps(info, indent=2))
print_items(original_items)

In [ ]:
# Check that all asset files exist (or not) in the s3 bucket
def check_assets(items: list[dict], exist: bool):
    for item in items:
        files = get_files(item)
        for s3_file in files:
            s3_bucket, s3_prefix = prefect_utils.get_s3_bucket(s3_file)
            s3_bucket.logger.setLevel(logging.WARNING)
            objects = s3_bucket.list_objects(s3_prefix, _sync=True)
            if exist:
                assert objects, f"{s3_file!r} is missing"
                print(f"{s3_file!r} exists")
            else:
                assert not objects, f"{s3_file!r} should have been removed"
                print(f"{s3_file!r} has been removed")
check_assets(original_items, True)

In [ ]:
# Compare two list of items
def compare(l1: list[dict], l2: list[dict]):
    len1 = len(l1)
    len2 = len(l2)
    assert len1 == len2, f"Lists have different lengths: {len1} vs {len2}"
    for i in range(len1):
        d1 = l1[i]
        d2 = l2[i]
        if d1 != d2:
            raise RuntimeError(
                f"Different values for item #{i}:\n{json.dumps(d1, indent=2)}\nVS\n{json.dumps(d2, indent=2)}")

In [ ]:
# For now, the items have not changed after triggering the lifecyle
# and GETting them from the catalog, because they are not expired.
pystac_items = catalog_client.search(
    owner_id=OWNER_ID,
    collections=[col_name],
    max_items=10e3,
).items
unchanged_items: list[dict] = items_to_dicts(pystac_items)
compare(original_items, unchanged_items)
print("For now, nothing has changed.")
print_items(unchanged_items)

In [ ]:
old_date: str = datetime(2000, 1, 1).strftime(DATETIME_FORMAT_MS)
expired_ids: list[str] = []
unchanged_ids: list[str] = []

# We advance one of two item expiration date to force its cleaning. 
for i in range(0, len(original_items), 2):
    expired_ids.append(original_items[i]["id"])

    # Update the stac item
    item = copy.deepcopy(original_items[i])
    item["properties"]["expires"] = old_date
    catalog_client.update_item(item)

# We leave the other items unchanged to check that they are not modified by the cleaning.
for j in range(1, len(original_items), 2):
    unchanged_ids.append(original_items[j]["id"])

In [ ]:
import time

def lifecycle_done(items):
    return all(
        item["properties"].get("unpublished") and not item.get("assets", {})
        for item in items
    )

def wait_for_lifecycle(expired_ids, timeout_s=120, poll_s=2):
    deadline = time.time() + timeout_s
    last_items = []

    while time.time() < deadline:
        pystac_items = catalog_client.search(
            owner_id=OWNER_ID,
            collections=[col_name],
            ids=expired_ids,
            max_items=10e3,
        ).items
        last_items = items_to_dicts(pystac_items)

        if lifecycle_done(last_items):
            return last_items

        time.sleep(poll_s)

    print_items(last_items)
    raise TimeoutError("Data lifecycle did not finish before timeout")

In [ ]:
def wait_for_deleted_assets(items: list[dict], timeout_s=120, poll_s=2):
    deadline = time.time() + timeout_s
    last_error = None

    while time.time() < deadline:
        try:
            check_assets(items, False)
            return
        except AssertionError as error:
            last_error = error
            time.sleep(poll_s)

    raise TimeoutError(f"Assets were not deleted before timeout: {last_error}")

if local_mode:
  # Wait until the expired items are actually unpublished and their assets are gone.
  unpublished_items = wait_for_lifecycle(expired_ids)
else:
  # And if we GET our expired items 
  pystac_items = catalog_client.search(
      owner_id=OWNER_ID,
      collections=[col_name],
      ids=expired_ids,
      max_items=10e3,
  ).items
  unpublished_items: list[dict] = items_to_dicts(pystac_items)

# We can see that the updated and unpublished fields have been set,
# and that the assets are missing.
print_items(unpublished_items)
for item in unpublished_items:
    assert item["properties"]["updated"]
    assert item["properties"]["unpublished"]
    assert not item["assets"]

    # Lifecycle should have refreshed the updated timestamp.
    assert (
        item["properties"]["updated"] != original_updated[item["id"]]
    )

    original_item = [i for i in original_items if i["id"] == item["id"]]
    wait_for_deleted_assets(original_item)

    # If we get the asset file paths from the original items, 
    # we can check that they were deleted from the bucket.
    original_item = [i for i in original_items if i["id"] == item["id"]]
    wait_for_deleted_assets(original_item)

In [ ]:
# On the other hand, the items that are not expired 
# were not changed by the data lifecycle
pystac_items = catalog_client.search(
    owner_id=OWNER_ID,
    collections=[col_name],
    ids=unchanged_ids,
    max_items=10e3,
).items
unchanged_items: list[dict] = items_to_dicts(pystac_items)

print_items(unchanged_items)
for item in unchanged_items:
    original_item = [i for i in original_items if i["id"] == item["id"]][0]
    assert item == original_item

    # We can check that the assets are still existing in the bucket.    
    check_assets([item], True)